<a href="https://colab.research.google.com/github/ChaitanyaDani5802/Gemini-AI-RAG-WebUI-Project/blob/main/Gemini_AI_RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to use Gemini file search tool as an easy RAG alternative

###1. Create a new folder template.

In [1]:
!rm -rf templates
!mkdir -p templates
print("Template's folder created successfully!!")

Template's folder created successfully!!


###2. Create using UI design HTML file.

In [2]:
%%writefile /content/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Gemini File Search RAG</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; }
        .container { background-color: #fff; padding: 20px; border-radius: 8px; box-shadow: 0 0 10px rgba(0,0,0,0.1); max-width: 800px; margin: auto; }
        h1, h3 { color: #333; }
        .upload-section, .query-section, .response-section { margin-bottom: 20px; border-bottom: 1px solid #eee; padding-bottom: 15px; }
        input[type="file"], input[type="submit"], button { padding: 10px 15px; border: none; border-radius: 5px; cursor: pointer; }
        input[type="file"] { background-color: #e9ecef; }
        input[type="submit"] { background-color: #007bff; color: white; }
        button { background-color: #28a745; color: white; margin-left: 10px; }
        textarea { width: 100%; padding: 10px; border-radius: 5px; border: 1px solid #ddd; margin-top: 10px; min-height: 100px; }
        #response { background-color: #e2e6ea; padding: 15px; border-radius: 5px; min-height: 50px; overflow-y: auto; }
    </style>
</head>
<body>
    <div class="container">
        <h1>Gemini File Search RAG Alternative</h1>

        <div class="upload-section">
            <h3>Upload Documents (PDF, TXT, DOCX)</h3>
            <form id="uploadForm" enctype="multipart/form-data">
                <input type="file" name="files" multiple>
                <input type="submit" value="Upload">
            </form>
            <p id="uploadStatus"></p>
        </div>

        <div class="query-section">
            <h3>Ask a Question</h3>
            <textarea id="queryInput" placeholder="Enter your question here..."></textarea>
            <button id="submitQuery">Submit Query</button>
        </div>

        <div class="response-section">
            <h3>Response</h3>
            <div id="response"></div>
        </div>
    </div>

    <script>
        // JavaScript will be added here later for interactivity
    </n>
</body>
</html>

Writing /content/templates/index.html


###3. Setup Pinggy Tunnel

####Install Pinggy

In [3]:
!pip install pinggy

#### Create Pinggy Tunnel

In [4]:
import pinggy
tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:8000"
)

try:
    # Accessing the first URL from the 'urls' attribute
    if tunnel1.urls:
        print(f"Tunnel1 started ~ URL: {tunnel1.urls[0]}")
    else:
        print("Tunnel1 started but no URLs found in tunnel1.urls.")
except AttributeError:
    print("Tunnel1 started. Could not find 'urls' attribute on the Tunnel object.")
    print(f"Available attributes on Tunnel object: {dir(tunnel1)}")

Tunnel1 started ~ URL: http://njuuj-35-198-238-238.free.pinggy.net


###4. Build the Alternative App to RAG: Study RAG

In [5]:
import os
from flask import Flask, request, render_template, jsonify
import google.generativeai as genai
import shutil

# --- Configuration ---
# Configure your Gemini API key
# It's recommended to store API keys securely, e.g., in environment variables or Colab secrets.
# For demonstration, you can replace 'YOUR_GEMINI_API_KEY' with your actual key.
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY") # Replace 'YOUR_GEMINI_API_KEY' if not using env variable
genai.configure(api_key=GEMINI_API_KEY)

# Initialize the Generative Model (e.g., 'gemini-pro')
model = genai.GenerativeModel('gemini-pro')

# --- Flask App Setup ---
app = Flask(__name__, template_folder='/content/templates')
UPLOAD_FOLDER = '/content/uploaded_docs'

# Ensure upload directory exists
if os.path.exists(UPLOAD_FOLDER):
    shutil.rmtree(UPLOAD_FOLDER)
os.makedirs(UPLOAD_FOLDER)

# --- Routes ---

@app.route('/')
def index():
    """Serves the main HTML page."""
    return render_template('index.html')

@app.route('/upload', methods=['POST'])
def upload_files():
    """Handles file uploads."""
    if 'files' not in request.files:
        return jsonify(success=False, message='No file part in the request'), 400

    uploaded_files = request.files.getlist('files')
    if not uploaded_files or uploaded_files[0].filename == '':
        return jsonify(success=False, message='No selected file'), 400

    file_paths = []
    for file in uploaded_files:
        if file.filename:
            filepath = os.path.join(UPLOAD_FOLDER, file.filename)
            file.save(filepath)
            file_paths.append(filepath)

    # In a real RAG system, you would process these files here:
    # - Extract text (e.g., using libraries like PyPDF2, python-docx)
    # - Chunk the text
    # - Create embeddings
    # - Store embeddings in a vector database
    print(f"Uploaded files: {file_paths}")
    return jsonify(success=True, message=f'Files uploaded and ready for processing: {file_paths}')

@app.route('/query', methods=['POST'])
def process_query():
    """Handles user queries and uses Gemini for response."""
    data = request.get_json()
    user_query = data.get('query')

    if not user_query:
        return jsonify(success=False, message='No query provided'), 400

    # In a full RAG implementation, you would:
    # 1. Retrieve relevant text chunks from your processed documents based on the user_query
    # 2. Construct a prompt that includes the user_query and the retrieved context
    #    e.g., prompt = f"Based on the following context: {context_from_docs}\n\nAnswer the question: {user_query}"

    # For now, a simple direct call to Gemini-pro
    try:
        response = model.generate_content(user_query)
        return jsonify(success=True, response=response.text)
    except Exception as e:
        return jsonify(success=False, message=f'Error generating response: {str(e)}'), 500

# --- Run the Flask app ---
# This will start the web server, accessible via the Pinggy tunnel.
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8000)


 * Serving Flask app '__main__'
 * Debug mode: off


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/Jun/2026 12:37:29] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Jun/2026 12:37:29] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [18/Jun/2026 12:40:24] "GET /?files=Claude+AI+Use+Cases+Facts.docx HTTP/1.1" 200 -
INFO:werkzeug:127.0